In [24]:
import pandas as pd 
import numpy as np 
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split      
from sklearn.metrics import roc_auc_score,accuracy_score,f1_score     

In [25]:
train = pd.read_csv("train.csv")
train.head()

,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,...,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,Gender,Customer Type,Type of Travel,Class,satisfaction
0,0,36,694,4,4,4,3,1,4,1,...,4,3,1,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
1,1,56,651,5,5,5,5,5,4,4,...,5,4,3,0,0.0,Male,Loyal Customer,Business travel,Business,True
2,2,31,1371,3,4,2,5,3,3,3,...,4,5,3,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
3,3,10,1616,3,5,3,3,3,3,3,...,3,4,3,0,0.0,Male,Loyal Customer,Personal Travel,Eco,False
4,4,23,481,3,4,3,4,5,3,5,...,5,3,5,5,0.0,Female,disloyal Customer,Business travel,Eco,False


In [26]:
test = pd.read_csv("test.csv")
test.head()

,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,...,Leg room service,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,Gender,Customer Type,Type of Travel,Class
0,699635,25,1197,4,4,4,2,3,4,3,...,5,2,1,3,0,0.0,Female,disloyal Customer,Business travel,Eco
1,699636,36,829,5,5,5,5,5,4,5,...,2,4,3,5,0,0.0,Male,Loyal Customer,Business travel,Eco
2,699637,45,472,4,5,5,5,4,4,4,...,3,1,3,4,0,0.0,Male,Loyal Customer,Business travel,Eco
3,699638,45,762,2,2,2,5,3,2,3,...,4,4,4,3,0,0.0,Female,disloyal Customer,Business travel,Business
4,699639,34,594,1,2,1,1,2,1,2,...,4,2,1,2,0,0.0,Male,Loyal Customer,Personal Travel,Eco Plus


In [27]:
train.shape

(699635, 23)

In [28]:
train.dtypes

id                                     int64
Age                                    int64
Flight Distance                        int64
Inflight wifi service                  int64
Departure/Arrival time convenient      int64
Ease of Online booking                 int64
Gate location                          int64
Food and drink                         int64
Online boarding                        int64
Seat comfort                           int64
Inflight entertainment                 int64
On-board service                       int64
Leg room service                       int64
Baggage handling                       int64
Checkin service                        int64
Cleanliness                            int64
Departure Delay in Minutes             int64
Arrival Delay in Minutes             float64
Gender                                object
Customer Type                         object
Type of Travel                        object
Class                                 object
satisfacti

In [29]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 699635 entries, 0 to 699634
Data columns (total 23 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   id                                 699635 non-null  int64  
 1   Age                                699635 non-null  int64  
 2   Flight Distance                    699635 non-null  int64  
 3   Inflight wifi service              699635 non-null  int64  
 4   Departure/Arrival time convenient  699635 non-null  int64  
 5   Ease of Online booking             699635 non-null  int64  
 6   Gate location                      699635 non-null  int64  
 7   Food and drink                     699635 non-null  int64  
 8   Online boarding                    699635 non-null  int64  
 9   Seat comfort                       699635 non-null  int64  
 10  Inflight entertainment             699635 non-null  int64  
 11  On-board service                   6996

In [30]:
train.describe()

,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes
count,699635.00000,699635.000000,699635.000000,699635.000000,699635.00000,699635.00000,699635.000000,699635.000000,699635.000000,699635.000000,699635.000000,699635.000000,699635.000000,699635.000000,699635.000000,699635.000000,699635.000000,699343.000000
mean,349817.00000,39.001928,1353.778702,2.778722,3.15621,2.81324,3.013734,3.265644,3.364156,3.568950,3.453910,3.517404,3.472847,3.805443,3.420291,3.380401,1.176209,1.134280
std,201967.37213,13.800943,954.422566,1.280419,1.47539,1.32955,1.254014,1.300202,1.288538,1.275333,1.308157,1.238474,1.271755,1.079175,1.210604,1.277870,5.982187,5.749304
min,0.00000,7.000000,67.000000,0.000000,0.00000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000
25%,174908.50000,27.000000,631.000000,2.000000,2.00000,2.00000,2.000000,2.000000,2.000000,3.000000,2.000000,3.000000,2.000000,3.000000,3.000000,2.000000,0.000000,0.000000
50%,349817.00000,40.000000,954.000000,3.000000,3.00000,3.00000,3.000000,3.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,0.000000,0.000000
75%,524725.50000,50.000000,1814.000000,4.000000,4.00000,4.00000,4.000000,4.000000,4.000000,5.000000,5.000000,4.000000,5.000000,5.000000,4.000000,4.000000,0.000000,0.000000
max,699634.00000,85.000000,4983.000000,5.000000,5.00000,5.00000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,489.000000,491.000000


In [31]:
train.isna().sum()

id                                     0
Age                                    0
Flight Distance                        0
Inflight wifi service                  0
Departure/Arrival time convenient      0
Ease of Online booking                 0
Gate location                          0
Food and drink                         0
Online boarding                        0
Seat comfort                           0
Inflight entertainment                 0
On-board service                       0
Leg room service                       0
Baggage handling                       0
Checkin service                        0
Cleanliness                            0
Departure Delay in Minutes             0
Arrival Delay in Minutes             292
Gender                                 0
Customer Type                          0
Type of Travel                         0
Class                                  0
satisfaction                           0
dtype: int64

In [32]:
train["Customer Type"].value_counts()

Customer Type
Loyal Customer       576990
disloyal Customer    122645
Name: count, dtype: int64

In [33]:
train["Gender"].value_counts()

Gender
Male      351683
Female    347952
Name: count, dtype: int64

In [34]:
train["Type of Travel"].value_counts()

Type of Travel
Business travel    497441
Personal Travel    202194
Name: count, dtype: int64

In [35]:
train["Class"].value_counts()

Class
Business    342212
Eco         327404
Eco Plus     30019
Name: count, dtype: int64

In [36]:
test_id = test["id"]
X=train.drop(columns=["id","satisfaction"],axis=1)
y=train["satisfaction"]
X_test=test.drop("id",axis=1)

In [37]:
y.value_counts()

satisfaction
False    389296
True     310339
Name: count, dtype: int64

In [38]:
y.astype(int)

0         0
1         1
2         0
3         0
4         0
         ..
699630    0
699631    0
699632    1
699633    1
699634    0
Name: satisfaction, Length: 699635, dtype: int32

In [39]:
y.nunique()

2

In [40]:
y.unique()

array([False,  True])

In [41]:
X["Customer Type"] = X["Customer Type"].str.replace("Customer","").str.replace("Customer","")
X["Customer Type"] = X["Customer Type"].astype(object)
X_test["Customer Type"] = X_test["Customer Type"].str.replace("Customer","").str.replace("Customer","")
X_test["Customer Type"] = X_test["Customer Type"].astype(object)
X["Type of Travel"] = X["Type of Travel"].str.replace("Travel","").str.replace("Travel","")
X["Type of Travel"] = X["Type of Travel"].astype(object)
X_test["Type of Travel"] = X_test["Type of Travel"].str.replace("Travel","").str.replace("Travel","")
X_test["Type of Travel"] = X_test["Type of Travel"].astype(object)

In [42]:
X.dtypes

Age                                    int64
Flight Distance                        int64
Inflight wifi service                  int64
Departure/Arrival time convenient      int64
Ease of Online booking                 int64
Gate location                          int64
Food and drink                         int64
Online boarding                        int64
Seat comfort                           int64
Inflight entertainment                 int64
On-board service                       int64
Leg room service                       int64
Baggage handling                       int64
Checkin service                        int64
Cleanliness                            int64
Departure Delay in Minutes             int64
Arrival Delay in Minutes             float64
Gender                                object
Customer Type                         object
Type of Travel                        object
Class                                 object
dtype: object

In [43]:
X_train,X_val,y_train,y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=2
)

cat_features = X_train.select_dtypes("object").columns.tolist()
num_features = X_train.select_dtypes(["int64","float64"]).columns.tolist()

med = X_train["Arrival Delay in Minutes"].median()
X_train["Arrival Delay in Minutes"] = X_train["Arrival Delay in Minutes"].fillna(med)
X_test["Arrival Delay in Minutes"] = X_test["Arrival Delay in Minutes"].fillna(med)

In [44]:
model = CatBoostClassifier(
    iterations=1500,
    learning_rate=0.03,
    loss_function="Logloss",
    cat_features=cat_features,
    verbose=200,
    devices="GPU",
    random_state=2
)

In [45]:
model.fit(X_train,y_train,cat_features=cat_features)

0:	learn: 0.6631323	total: 248ms	remaining: 6m 12s
200:	learn: 0.2398512	total: 57.1s	remaining: 6m 9s
400:	learn: 0.2327136	total: 1m 55s	remaining: 5m 16s
600:	learn: 0.2301681	total: 2m 52s	remaining: 4m 17s
800:	learn: 0.2284768	total: 3m 40s	remaining: 3m 12s
1000:	learn: 0.2273196	total: 4m 28s	remaining: 2m 14s
1200:	learn: 0.2264222	total: 5m 24s	remaining: 1m 20s
1400:	learn: 0.2256242	total: 6m 16s	remaining: 26.6s
1499:	learn: 0.2252528	total: 6m 49s	remaining: 0us


CatBoostClassifier(cat_features=['Gender', 'Customer Type', 'Type of Travel', 'Class'], devices='GPU', iterations=1500, learning_rate=0.03, loss_function='Logloss', random_state=2, verbose=200)

In [46]:
y_pred = model.predict_proba(X_val)[:,1]
roc = roc_auc_score(y_val,y_pred)
print(roc)

0.9577126287740002


In [48]:
pred = model.predict_proba(X_test)[:,1]

submission = pd.DataFrame({
    "id":test_id,
    "satisfaction": pred
})

In [49]:
submission.to_csv("submission.csv",index = False)
submission.head()

,id,satisfaction
0,699635,0.267709
1,699636,0.894821
2,699637,0.879041
3,699638,0.056272
4,699639,0.063862
